# Responsibilities Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook reads the LLM-filtered AI engineer job postings and uses `gpt-5.4-mini` to summarize the responsibilities that consistently appear across the roles.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

### Load classified jobs from file

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(classified_jobs_path, lines=True)
print(f"Loaded {len(jobs_for_summary)} classified job postings")

### Filter out non AI-Engineer Jobs

In [ ]:
# Keep only the jobs the LLM classified as true AI engineering roles
jobs_for_summary = jobs_for_summary[jobs_for_summary["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_for_summary)} classified AI engineering job postings")

### Merge all descriptions into a single String

In [ ]:
descriptions = jobs_for_summary["description"].astype(str)
job_descriptions = "\n\n---\n\n".join(descriptions)

print(job_descriptions[:10000] + "\n\n...")

### Prepare the prompt

We combine the job descriptions into one prompt and ask the model to summarize the responsibilities that show up repeatedly.

In [ ]:
prompt = f"""
Read the job descriptions below and summarize the responsibilities that appear repeatedly across them.

Return markdown in exactly this format:
### Shared responsibilities
- 5 concise bullet points

Requirements:
- Focus only on common responsibilities
- Ignore benefits, recruiting language, company marketing, and hiring process details
- Keep bullets concrete and specific
- Do not mention company names

Each job description is separated by "---".

Job descriptions:
{job_descriptions}
""".strip()

print("Prompt for summarization: \n")
print(prompt[:1000])
print(f"\nPrompt character count: {len(prompt):,}")

### Can the model handle our prompt?

https://developers.openai.com/

https://developers.openai.com/api/docs/models/gpt-5.4-mini

https://platform.openai.com/tokenizer

https://ollama.com/library/gemma4

https://ollama.com/library/gemma3

In [ ]:
import tiktoken

# gpt-5.4-mini uses the o200k_base encoding (same family as GPT-4o/GPT-5)
encoding = tiktoken.get_encoding("o200k_base")

prompt_tokens = len(encoding.encode(prompt))

print(f"Prompt tokens: {prompt_tokens:,}")

### Calculate the price of the request

In [ ]:
input_price_per_1m_tokens = 2.5  # $ per 1M input tokens
prompt_cost = (prompt_tokens / 1_000_000) * input_price_per_1m_tokens

print(f"Estimated prompt cost: ${prompt_cost:.4f}")

The prompt is large because it includes all filtered job descriptions. Counting tokens before the API call helps us estimate cost and check whether the prompt fits into the model context window.

### Summarize the responsibilities

We send the prompt to the model and display the response as markdown.

In [ ]:
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt,
)

display(Markdown(response.output_text))